In [1]:
import os
import numpy as np
import pandas as pd
from ogcore.utils import safe_read_pickle
from ogcore.output_tables import dynamic_revenue_decomposition

In [2]:
CUR_DIR = './'
example_dir = 'TCJA_TMD##frisch##0.4##zeta_D##0.4##g_y_annual##0.02##tG1##20/'
base_dir = os.path.join(CUR_DIR, example_dir, "OUTPUT_BASELINE")
reform_dir = os.path.join(CUR_DIR, example_dir, "OUTPUT_REFORM")

base_tpi = safe_read_pickle(os.path.join(base_dir, "TPI", "TPI_vars.pkl"))
base_params = safe_read_pickle(os.path.join(base_dir, "model_params.pkl"))
base_ss = safe_read_pickle(os.path.join(base_dir, "SS", "SS_vars.pkl"))
reform_tpi = safe_read_pickle(os.path.join(reform_dir, "TPI", "TPI_vars.pkl"))
reform_params = safe_read_pickle(os.path.join(reform_dir, "model_params.pkl"))
reform_ss = safe_read_pickle(os.path.join(reform_dir, "SS", "SS_vars.pkl"))

In [3]:
df = dynamic_revenue_decomposition(base_params, base_tpi, base_ss, reform_params, reform_tpi, reform_ss, start_year=2026, num_years=9, full_break_out=True)
df

Year,Variable,2026,2027,2028,2029,2030,2031,2032,2033,2034,2026-2034,SS
0,IIT: Pct Change due to tax rates,-7.15,-7.15,-7.15,-7.15,-7.15,-7.15,-7.15,-7.15,-7.15,-7.15,-7.15
1,IIT: Pct Change due to behavior,1.10,1.11,1.13,1.14,1.16,1.17,1.18,1.19,1.20,1.15,1.44
2,IIT: Pct Change due to macro,0.01,-0.01,-0.03,-0.05,-0.08,-0.10,-0.12,-0.15,-0.18,-0.08,-0.08
3,IIT: Overall Pct Change in taxes,-6.12,-6.13,-6.13,-6.13,-6.14,-6.15,-6.17,-6.18,-6.20,-6.15,-5.89
4,CIT: Pct Change due to tax rates,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,CIT: Pct Change due to behavior,0.18,0.34,0.49,0.57,0.63,0.65,0.65,0.63,0.58,0.52,3.57
6,CIT: Pct Change due to macro,1.05,0.83,0.64,0.51,0.41,0.34,0.30,0.28,0.28,0.51,-2.24
7,CIT: Overall Pct Change in taxes,1.23,1.17,1.13,1.08,1.04,0.99,0.95,0.91,0.86,1.04,1.25
8,All: Pct Change due to tax rates,-6.77,-6.77,-6.77,-6.77,-6.77,-6.77,-6.77,-6.77,-6.77,-6.77,-6.78
9,All: Pct Change due to behavior,1.05,1.07,1.10,1.11,1.13,1.14,1.15,1.16,1.16,1.12,1.56


In [4]:
# Now apply these percentage changes to the baseline revenue
# Take CBO baseline (to include not just IIT)
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([5.394, 5.756, 5.944, 6.133, 6.354, 6.661, 6.899, 7.176, 7.459])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[8:, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 9), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
8,Rev Change Due to Tax Rates,-0.37,-0.39,-0.40,-0.42,-0.43,-0.45,-0.47,-0.49,-0.50,-3.91
9,Rev Change Due to Behavior,0.06,0.06,0.07,0.07,0.07,0.08,0.08,0.08,0.09,0.65
10,Rev Change Due to Macro,0.00,0.00,0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.03
11,Total Revenue Change,-0.31,-0.33,-0.34,-0.35,-0.37,-0.39,-0.40,-0.42,-0.43,-3.34


In [5]:
# Get level changes just for IIT + Payroll
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([4.655, 5.007, 5.184, 5.365, 5.573, 5.783, 5.994, 6.227, 6.476])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 9), (df_levels.shape[0], 1)) / 100
df_levels["2026-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2026,2027,2028,2029,2030,2031,2032,2033,2034,2026-2034
0,Rev Change Due to Tax Rates,-0.33,-0.36,-0.37,-0.38,-0.40,-0.41,-0.43,-0.44,-0.46,-3.59
1,Rev Change Due to Behavior,0.05,0.06,0.06,0.06,0.06,0.07,0.07,0.07,0.08,0.58
2,Rev Change Due to Macro,0.00,-0.00,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.04
3,Total Revenue Change,-0.28,-0.31,-0.32,-0.33,-0.34,-0.36,-0.37,-0.39,-0.40,-3.09


In [6]:
result_df_static = pd.read_csv('../../Tax-Calculator-thru74/tax-brain-result-static.csv', index_col = 0)
result_df_dynamic = pd.read_csv('../../Tax-Calculator-thru74/tax-brain-result-dynamic.csv', index_col = 0)

In [7]:
# Or we can use the Tax-Calc baseline for a direct comparison
base_revenue = result_df_static.loc["Base", ['2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 9), (df_levels.shape[0], 1)) / 100
df_levels["2026-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2026,2027,2028,2029,2030,2031,2032,2033,2034,2026-2034
0,Rev Change Due to Tax Rates,-0.32,-0.34,-0.35,-0.36,-0.38,-0.40,-0.41,-0.43,-0.44,-3.43
1,Rev Change Due to Behavior,0.05,0.05,0.06,0.06,0.06,0.06,0.07,0.07,0.07,0.56
2,Rev Change Due to Macro,0.00,-0.00,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.04
3,Total Revenue Change,-0.28,-0.29,-0.30,-0.31,-0.33,-0.34,-0.36,-0.37,-0.39,-2.96


In [8]:
# jason's get-around

df_levels = df.loc[0:3, df.columns[:-2]]
tc_diff = result_df_static.loc["Difference", ['2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
tc_reform = result_df_static.loc["Reform", ['2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
df_levels.loc[0, df_levels.columns[1:]] = tc_diff
df_levels.loc[1, df_levels.columns[1:]] = (df.loc[1, df_levels.columns[1:]] / 100) * tc_reform
df_levels.loc[2, df_levels.columns[1:]] = (df.loc[2, df_levels.columns[1:]] / 100) * df_levels.loc[1, df_levels.columns[1:]]
df_levels.loc[3, df_levels.columns[1:]] = df_levels.loc[0, df_levels.columns[1:]] + df_levels.loc[1, df_levels.columns[1:]] + df_levels.loc[2, df_levels.columns[1:]]
df_levels["2026-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2026,2027,2028,2029,2030,2031,2032,2033,2034,2026-2034
0,Rev Change Due to Tax Rates,-0.28,-0.29,-0.30,-0.31,-0.32,-0.33,-0.34,-0.34,-0.35,-2.86
1,Rev Change Due to Behavior,0.05,0.05,0.05,0.05,0.06,0.06,0.06,0.07,0.07,0.52
2,Rev Change Due to Macro,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00
3,Total Revenue Change,-0.24,-0.24,-0.25,-0.25,-0.26,-0.27,-0.27,-0.28,-0.28,-2.34


In [9]:
# jason's get-around, modified

df_levels = df.loc[0:3, df.columns[:-2]]
tc_diff = result_df_static.loc["Difference", ['2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
tc_base = result_df_static.loc["Base", ['2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
df_levels.loc[0, df_levels.columns[1:]] = tc_diff
df_levels.loc[1, df_levels.columns[1:]] = (df.loc[1, df_levels.columns[1:]] / 100) * tc_base
df_levels.loc[2, df_levels.columns[1:]] = (df.loc[2, df_levels.columns[1:]] / 100) * tc_base
df_levels.loc[3, df_levels.columns[1:]] = df_levels.loc[0, df_levels.columns[1:]] + df_levels.loc[1, df_levels.columns[1:]] + df_levels.loc[2, df_levels.columns[1:]]
df_levels["2026-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2026,2027,2028,2029,2030,2031,2032,2033,2034,2026-2034
0,Rev Change Due to Tax Rates,-0.28,-0.29,-0.30,-0.31,-0.32,-0.33,-0.34,-0.34,-0.35,-2.86
1,Rev Change Due to Behavior,0.05,0.05,0.06,0.06,0.06,0.06,0.07,0.07,0.07,0.56
2,Rev Change Due to Macro,0.00,-0.00,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.04
3,Total Revenue Change,-0.23,-0.24,-0.25,-0.25,-0.26,-0.27,-0.28,-0.28,-0.29,-2.35


In [10]:
df_levels = df.loc[0:3, df.columns[:-2]]
tc_diff = result_df_dynamic.loc["Difference", ['2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
tc_base = result_df_dynamic.loc["Base", ['2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
df_levels.loc[0, df_levels.columns[1:]] = tc_diff
df_levels.loc[1, df_levels.columns[1:]] = (df.loc[1, df_levels.columns[1:]] / 100) * tc_base
df_levels.loc[2, df_levels.columns[1:]] = (df.loc[2, df_levels.columns[1:]] / 100) * tc_base
df_levels.loc[3, df_levels.columns[1:]] = df_levels.loc[0, df_levels.columns[1:]] + df_levels.loc[1, df_levels.columns[1:]] + df_levels.loc[2, df_levels.columns[1:]]
df_levels["2026-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2026,2027,2028,2029,2030,2031,2032,2033,2034,2026-2034
0,Rev Change Due to Tax Rates,-0.22,-0.23,-0.23,-0.24,-0.24,-0.25,-0.26,-0.26,-0.27,-2.20
1,Rev Change Due to Behavior,0.05,0.05,0.06,0.06,0.06,0.06,0.07,0.07,0.07,0.56
2,Rev Change Due to Macro,0.00,-0.00,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.04
3,Total Revenue Change,-0.17,-0.17,-0.18,-0.18,-0.19,-0.19,-0.20,-0.20,-0.20,-1.68
